In [1]:
import os
import logging
import time
from pathlib import Path
from typing import List, Optional

import polars as pl
import h5py
import numpy as np
import json

DEFAULT_H5_PATH = Path("../data/h5datasets/baikal_mc_merged.h5")
TRAIN_PARTS_PATH = Path("./default_parts.json")

In [7]:
# Find parts compliment to the train ones

def read_parts(hf, particle) -> list:
    group = hf[f'{particle}']
    parts = list(group['raw/data'].keys())
    return sorted(parts, key=lambda x: eval(x.split('_')[-1]))

parts_dict_all = {}
with h5py.File(DEFAULT_H5_PATH) as hf:
    for ptype in ['muatm_2020', 'nuatm_2020', 'nue2_2020']:
        parts = read_parts(hf, ptype)
        parts_dict_all[ptype.replace('_2020', '')] = [int(name.replace('part_', "")) for name in parts]

with open(TRAIN_PARTS_PATH) as f:
    parts_dict_train = json.load(f)

parts_dict_test = {}
for ptype, parts in parts_dict_all.items():
    train_parts = parts_dict_train[ptype]
    print(f"{ptype}: {len(train_parts)=}")
    test_parts = list(set(parts) - set(train_parts))
    test_parts = sorted(test_parts)
    parts_dict_test[ptype] = test_parts
    print(f"{ptype}: {len(test_parts)=}")

muatm: len(train_parts)=432
muatm: len(test_parts)=19572
nuatm: len(train_parts)=187
nuatm: len(test_parts)=1813
nue2: len(train_parts)=333
nue2: len(test_parts)=67


In [8]:
"""
Sample 1000 parts for muatm, 100 parts per each nu type
and save to 'testds_parts.json'
"""

import random
sampled_test_json = {}

sampled_test_json['muatm'] =  sorted(random.sample(parts_dict_test['muatm'], 10000))
sampled_test_json['nuatm'] =  sorted(random.sample(parts_dict_test['nuatm'], min(100, len(parts_dict_test['nuatm']))))
sampled_test_json['nue2'] =  sorted(random.sample(parts_dict_test['nue2'], min(100, len(parts_dict_test['nue2']))))

with open(TRAIN_PARTS_PATH.parent / 'testds_parts.json', "w") as f:
        json.dump(sampled_test_json, f, indent=4)